### ExpiryCounter

In [ ]:
import time
from collections import defaultdict, deque

class ExpiryCounter:
    def __init__(self, t: float):
        self.t = t
        self.global_queue = deque()       # (timestamp, key)
        self.key_counts = defaultdict(int) # key -> active count
        self.total = 0

    def _cleanup(self):
        now = time.time()
        while self.global_queue and now - self.global_queue[0][0] >= self.t:
            _, key = self.global_queue.popleft()
            self.key_counts[key] -= 1
            self.total -= 1

    def put_element(self, k):
        now = time.time()
        self.global_queue.append((now, k))
        self.key_counts[k] += 1
        self.total += 1

    def get_element_count(self, k):
        self._cleanup()
        return self.key_counts[k]

    def get_total_elements(self):
        self._cleanup()
        return self.total


# ---------- Test ----------
c = ExpiryCounter(t=2)

c.put_element("a")
c.put_element("b")
c.put_element("a")
print(f"count(a) = {c.get_element_count('a')}")   # 2
print(f"count(b) = {c.get_element_count('b')}")   # 1
print(f"total    = {c.get_total_elements()}")      # 3

time.sleep(2)

print("\n--- after 2s expiry ---")
print(f"count(a) = {c.get_element_count('a')}")   # 0
print(f"count(b) = {c.get_element_count('b')}")   # 0
print(f"total    = {c.get_total_elements()}")      # 0

c.put_element("c")
print(f"\ncount(c) = {c.get_element_count('c')}")  # 1
print(f"total    = {c.get_total_elements()}")      # 1

### Randomized Set

In [ ]:
from random import choice

class RandomizedSet:
    def __init__(self):
        self.list = []
        self.hashmap = {}

    def add(self, num):
        if num in self.hashmap:
            return False
        self.hashmap[num] = len(self.list)
        self.list.append(num)
        return True
    
    def remove(self, num):
        if num not in self.hashmap:
            return False
        last = self.list[-1]
        numindex = self.hashmap[num]

        # Update indexes
        self.list[numindex] = last
        self.hashmap[last] = numindex

        # Delete elements
        self.list.pop()
        del self.hashmap[num]

        return True
    
    def random(self):
        return choice(self.list)

### Time Based Key Value Store

In [ ]:
class TimeValue:
    def __init__(self):
        self.store: dict[str, list[str, str]] = {}

    def set(self, key, value, timestamp) -> None:
        if key not in self.store:
            self.store[key] = []
        self.store[key].append([value, timestamp])

    def get(self, key, timestamp):
        res = ''
        values = self.store.get(key, [])
        l, r = 0, len(values) - 1
        while (l <= r):
            # when l == r, 1 candidate remains that needs to be examined.
            #
            # Example:
            # values = [
            #     ["A", 1],
            #     ["B", 4],
            #     ["C", 7],
            # ]
            #
            # timestamp = 7
            # l=0, r=2, m=1 -> timestamp 4 is valid
            # res="B", l=2
            #
            # l=2, r=2      -> one candidate remains
            # m=2            -> timestamp 7 is valid
            # res="C", l=3
            m = (l + r) // 2
            if values[m][1] <= timestamp:
                res = values[m][0]
                l = m + 1
            else:
                r = m - 1
        return res

    def get2(self, key, timestamp):
        from bisect import bisect_right

        values = self.store.get(key, [])
        index = bisect_right(
            values,
            timestamp,
            key = lambda x:x[1]
        )

        return values[index - 1][0] if index > 0 else ''
